# Legal Clause Classification with CUAD

This notebook downloads the original CUAD dataset from Hugging Face, prepares a flattened clause-level table, and walks through exploratory data analysis on the extracted data.


In [ ]:
# Section 0 — Install & imports
# ── Run this cell first, every session ───────────────────────────────────────
# Clones the repo so all .py modules are available without manual file upload.

import sys, os

REPO_URL    = "https://github.com/pranaysamineni00/BT5153.git"
REPO_BRANCH = "main"
REPO_DIR    = "/content/BT5153"

if not os.path.exists(REPO_DIR):
    ret = os.system(f"git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
    if ret != 0:
        raise RuntimeError(f"git clone failed (exit {ret}). Check REPO_URL and REPO_BRANCH.")
else:
    os.system(f"git -C {REPO_DIR} pull --ff-only")   # keep in sync on re-runs

# Verify expected modules are present
expected = ["data_loading.py", "preprocessing.py", "training.py", "evaluation.py"]
missing  = [f for f in expected if not os.path.exists(os.path.join(REPO_DIR, f))]
if missing:
    print("Files in REPO_DIR:", os.listdir(REPO_DIR))
    raise FileNotFoundError(f"Missing modules after clone: {missing}")

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
os.makedirs("figures", exist_ok=True)

%pip install -q pandas pyarrow huggingface_hub matplotlib seaborn torch transformers \
    scikit-learn accelerate datasets

import importlib
import data_loading, preprocessing, training, evaluation

for mod in [data_loading, preprocessing, training, evaluation]:
    importlib.reload(mod)

from data_loading import load_cuad, load_ledgar
from preprocessing import (
    filter_clauses, plot_clause_frequency, build_clause_mappings,
    build_contract_records, split_contract_records, build_chunk_examples,
    MultiLabelChunkDataset, prepare_chunked_splits, sample_contracts, AutoTokenizer,
    aggregate_contract_predictions,
)
from training import (
    train_tfidf_lr, train_bert_ledgar_cuad, train_bert_cuad,
    train_legal_bert_cuad, train_longformer_cuad, train_legalbert_longformer_cuad,
    collect_logits_and_labels, choose_device,
)
from evaluation import (
    tune_per_clause_thresholds, compute_per_clause_metrics,
    compute_aggregate_metrics, plot_confusion_matrix, plot_model_comparison,
)
from IPython.display import display
import numpy as np
import pandas as pd


In [ ]:
from pathlib import Path

from huggingface_hub import hf_hub_download

raw_json_path = Path("data/cuad/CUAD_v1/CUAD_v1.json")
raw_json_path.parent.mkdir(parents=True, exist_ok=True)

if not raw_json_path.exists():
    downloaded_path = hf_hub_download(
        repo_id="theatticusproject/cuad",
        repo_type="dataset",
        filename="CUAD_v1/CUAD_v1.json",
        local_dir="data/cuad",
    )
    raw_json_path = Path(downloaded_path)

raw_json_path


In [ ]:
import json

with raw_json_path.open() as f:
    cuad = json.load(f)

documents = cuad["data"]
version = cuad.get("version")

print(f"CUAD version: {version}")
print(f"Number of contracts: {len(documents)}")
print(f"Paragraph keys: {list(documents[0]['paragraphs'][0].keys())}")
print(f"Questions per first contract: {len(documents[0]['paragraphs'][0]['qas'])}")


In [ ]:
import pandas as pd

rows = []

for doc in documents:
    for paragraph in doc["paragraphs"]:
        context = paragraph["context"]
        for qa in paragraph["qas"]:
            answers = qa.get("answers", [])
            rows.append(
                {
                    "contract_title": doc["title"],
                    "clause_type": qa["id"].split("__", 1)[-1],
                    "question": qa["question"],
                    "contract_text": context,
                    "is_impossible": qa["is_impossible"],
                    "has_answer": bool(answers),
                    "answer_count": len(answers),
                    "answer_texts": [answer["text"] for answer in answers],
                    "answer_starts": [answer["answer_start"] for answer in answers],
                }
            )

cuad_df = pd.DataFrame(rows)
flat_path = Path("data/cuad/cuad_qa_rows.parquet")
cuad_df.to_parquet(flat_path, index=False)

print(cuad_df.shape)
flat_path


In [ ]:

cuad_df

In [ ]:
cuad_df['question'][100]

## EDA

The cells below summarize class balance, contract length, clause coverage, and answer span behavior in the extracted CUAD table.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="talk")

cuad_df["contract_char_count"] = cuad_df["contract_text"].str.len()
cuad_df["contract_word_count"] = cuad_df["contract_text"].str.split().str.len()
cuad_df["question_word_count"] = cuad_df["question"].str.split().str.len()
cuad_df["answer_char_total"] = cuad_df["answer_texts"].apply(lambda answers: sum(len(answer) for answer in answers))

contract_df = cuad_df[["contract_title", "contract_char_count", "contract_word_count"]].drop_duplicates()

clause_summary = (
    cuad_df.groupby("clause_type")
    .agg(
        total_rows=("clause_type", "size"),
        positive_rows=("has_answer", "sum"),
        positive_rate=("has_answer", "mean"),
        avg_answer_count=("answer_count", "mean"),
    )
    .sort_values(["positive_rows", "positive_rate"], ascending=[False, False])
)

contract_positive_counts = (
    cuad_df.groupby("contract_title")["has_answer"]
    .sum()
    .reset_index(name="positive_clause_count")
    .merge(contract_df, on="contract_title", how="left")
)

answer_spans_df = cuad_df.explode(["answer_texts", "answer_starts"], ignore_index=True)
answer_spans_df = answer_spans_df[answer_spans_df["answer_texts"].notna()].copy()
answer_spans_df["answer_span_word_count"] = answer_spans_df["answer_texts"].str.split().str.len()
answer_spans_df["answer_span_char_count"] = answer_spans_df["answer_texts"].str.len()


In [ ]:
print(f"Clause-level rows: {len(cuad_df):,}")
print(f"Contracts: {contract_df['contract_title'].nunique():,}")
print(f"Clause types: {cuad_df['clause_type'].nunique():,}")
print(f"Positive rate: {cuad_df['has_answer'].mean():.2%}")
print(f"Average answers per row: {cuad_df['answer_count'].mean():.2f}")
print(f"Average words per contract: {contract_df['contract_word_count'].mean():,.0f}")

display(cuad_df[["has_answer", "answer_count", "contract_word_count", "question_word_count", "answer_char_total"]].describe())
display(clause_summary.head(10))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

overall_balance = cuad_df["has_answer"].value_counts().rename(index={True: "Has answer", False: "No answer"})
sns.barplot(x=overall_balance.index, y=overall_balance.values, ax=axes[0], palette="Set2")
axes[0].set_title("Overall Label Balance")
axes[0].set_xlabel("")
axes[0].set_ylabel("Rows")

sns.histplot(cuad_df["answer_count"], bins=15, discrete=True, ax=axes[1], color="#2a9d8f")
axes[1].set_title("Answers per Clause Query")
axes[1].set_xlabel("Answer count")
axes[1].set_ylabel("Rows")

plt.tight_layout()


In [ ]:
top_positive_counts = clause_summary.head(15).sort_values("positive_rows")
top_positive_rates = clause_summary.sort_values("positive_rate", ascending=False).head(15).sort_values("positive_rate")

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

sns.barplot(data=top_positive_counts, x="positive_rows", y=top_positive_counts.index, ax=axes[0], palette="crest")
axes[0].set_title("Top Clause Types by Positive Examples")
axes[0].set_xlabel("Positive rows")
axes[0].set_ylabel("Clause type")

sns.barplot(data=top_positive_rates, x="positive_rate", y=top_positive_rates.index, ax=axes[1], palette="flare")
axes[1].set_title("Top Clause Types by Positive Rate")
axes[1].set_xlabel("Positive rate")
axes[1].set_ylabel("Clause type")

plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(contract_df["contract_word_count"], bins=30, ax=axes[0], color="#457b9d")
axes[0].set_title("Distribution of Contract Length")
axes[0].set_xlabel("Words per contract")
axes[0].set_ylabel("Number of contracts")

sns.scatterplot(
    data=contract_positive_counts,
    x="contract_word_count",
    y="positive_clause_count",
    ax=axes[1],
    alpha=0.7,
    color="#e76f51",
)
axes[1].set_title("Contract Length vs Positive Clause Count")
axes[1].set_xlabel("Words per contract")
axes[1].set_ylabel("Positive clause labels")

plt.tight_layout()


In [ ]:
avg_answers_per_clause = clause_summary.sort_values("avg_answer_count", ascending=False).head(15).sort_values("avg_answer_count")

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.histplot(answer_spans_df["answer_span_word_count"], bins=30, ax=axes[0], color="#264653")
axes[0].set_title("Answer Span Length Distribution")
axes[0].set_xlabel("Words per answer span")
axes[0].set_ylabel("Number of answer spans")

sns.barplot(data=avg_answers_per_clause, x="avg_answer_count", y=avg_answers_per_clause.index, ax=axes[1], palette="mako")
axes[1].set_title("Clause Types with the Most Answers per Query")
axes[1].set_xlabel("Average answers per row")
axes[1].set_ylabel("Clause type")

plt.tight_layout()


In [ ]:
clause_summary.sort_values(["positive_rate", "positive_rows"], ascending=[False, False])


## Section 1 — Load Datasets

In [ ]:
# Section 1 — Load datasets
cuad_df = load_cuad("data/cuad")
ledgar  = load_ledgar("data/ledgar")


## Section 2 — EDA, Clause Filtering & Chunking

In [ ]:
# Section 2 — EDA + clause filtering + chunking
# ── Runtime configuration ────────────────────────────────────────────────────
# DEV_MODE=True  → 20% of contracts, 1 epoch, capped LEDGAR batches (~40 min total)
# DEV_MODE=False → full dataset, 3 epochs (original settings, ~5 hours total)
# Flip to False for the final submission run.
DEV_MODE = True

SAMPLE_FRAC        = 0.20 if DEV_MODE else 1.0
TRAIN_EPOCHS       = 1    if DEV_MODE else 3
LEDGAR_MAX_BATCHES = 100  if DEV_MODE else None  # caps LEDGAR warm-up to ~1,600 examples

# 2a. Bar chart — positive rate per clause type
import os; os.makedirs("figures", exist_ok=True)
clause_summary = plot_clause_frequency(cuad_df, save_path="figures/clause_frequency.png")
display(clause_summary.head(20))

# 2b. Filter clauses below threshold
MIN_POSITIVES = 20
filtered_df, excluded_clauses = filter_clauses(cuad_df, min_positives=MIN_POSITIVES)
print(f"\nRetained clause types: {filtered_df['clause_type'].nunique()}")
print(f"Excluded: {list(excluded_clauses.keys())}")

# 2c. Prepare BERT-family chunks (512 tokens, stride 128)
BERT_SPLITS = prepare_chunked_splits(
    filtered_df, model_name="bert-base-uncased", max_length=512, stride=128, seed=42,
    sample_frac=SAMPLE_FRAC,
)
print(f"Train chunks: {len(BERT_SPLITS['train_examples']):,} | "
      f"Val: {len(BERT_SPLITS['val_examples']):,} | "
      f"Test: {len(BERT_SPLITS['test_examples']):,}")

# 2d. Prepare Longformer chunks (4096 tokens, stride 512)
# sample_contracts is applied before splitting so Longformer trains on the same
# fraction of contracts as the BERT models, ensuring a fair comparison.
LF_TOKENIZER = AutoTokenizer.from_pretrained("allenai/longformer-base-4096", use_fast=True)
clause_to_id, id_to_clause = build_clause_mappings(filtered_df)
contract_records = build_contract_records(filtered_df)
contract_records = sample_contracts(contract_records, frac=SAMPLE_FRAC, seed=42)
train_r, val_r, test_r = split_contract_records(contract_records, seed=42)

lf_train_ex = build_chunk_examples(train_r, clause_to_id, LF_TOKENIZER, max_length=4096, stride=512)
lf_val_ex   = build_chunk_examples(val_r,   clause_to_id, LF_TOKENIZER, max_length=4096, stride=512)
lf_test_ex  = build_chunk_examples(test_r,  clause_to_id, LF_TOKENIZER, max_length=4096, stride=512)

LF_SPLITS = {
    "tokenizer":       LF_TOKENIZER,
    "id_to_clause":    id_to_clause,
    "train_examples":  lf_train_ex,
    "val_examples":    lf_val_ex,
    "test_examples":   lf_test_ex,
    "train_dataset":   MultiLabelChunkDataset(lf_train_ex),
    "val_dataset":     MultiLabelChunkDataset(lf_val_ex),
    "test_dataset":    MultiLabelChunkDataset(lf_test_ex),
    "train_records":   train_r,
    "val_records":     val_r,
    "test_records":    test_r,
}
print(f"Longformer train chunks: {len(lf_train_ex):,} | Val: {len(lf_val_ex):,} | Test: {len(lf_test_ex):,}")


## Section 3 — Train All Models

In [ ]:
# Section 3a — Hypothesis statement + TF-IDF + LR baseline
# ── Hypothesis statement ──────────────────────────────────────────────────────
print("""
HYPOTHESES:
H1: Legal-BERT will outperform BERT on clause types with specialised legal vocabulary
    (governing law, dispute resolution, indemnification) due to legal-domain pretraining.
H2: Longformer will outperform BERT-family on clause types where relevant language is
    spread across the full document (force majeure, termination for convenience),
    because it processes up to 4,096 tokens without chunking.
H3: Legal-BERT warm-started Longformer will be the strongest overall, combining
    domain-specific vocabulary with architectural long-context advantage.
""")

import numpy as np

# Count positive training examples per clause for reporting
label_matrix = np.array([ex["labels"] for ex in BERT_SPLITS["train_examples"]])
n_positive_train = {i: int(label_matrix[:, i].sum()) for i in range(label_matrix.shape[1])}

# 3a. TF-IDF + LR baseline
train_titles = {r["contract_title"] for r in BERT_SPLITS["train_records"]}
val_titles   = {r["contract_title"] for r in BERT_SPLITS["val_records"]}
train_df_subset = filtered_df[filtered_df["contract_title"].isin(train_titles)]
val_df_subset   = filtered_df[filtered_df["contract_title"].isin(val_titles)]

artifacts_tfidf = train_tfidf_lr(train_df_subset, val_df_subset, BERT_SPLITS["id_to_clause"])


In [ ]:
# Section 3b — BERT (CUAD only)
# batch_size=16 enabled by fp16 mixed precision; ~2x fewer gradient steps vs batch_size=8
artifacts_bert = train_bert_cuad(
    BERT_SPLITS["train_dataset"], BERT_SPLITS["val_dataset"],
    BERT_SPLITS["train_examples"], "bert-base-uncased",
    BERT_SPLITS["tokenizer"], BERT_SPLITS["id_to_clause"],
    epochs=TRAIN_EPOCHS, batch_size=16,
)


In [ ]:
# Section 3c — BERT (LEDGAR → CUAD)
# ledgar_epochs=1: one pass is enough for warm-starting; saves ~2/3 of Phase-1 wall time
# ledgar_max_batches caps Phase 1 in DEV_MODE so LEDGAR does not dominate wall time
artifacts_bert_ledgar = train_bert_ledgar_cuad(
    ledgar_dataset=ledgar,
    train_dataset=BERT_SPLITS["train_dataset"], val_dataset=BERT_SPLITS["val_dataset"],
    train_examples=BERT_SPLITS["train_examples"],
    model_name="bert-base-uncased", tokenizer=BERT_SPLITS["tokenizer"],
    id_to_clause=BERT_SPLITS["id_to_clause"],
    ledgar_epochs=1, ledgar_max_batches=LEDGAR_MAX_BATCHES,
    cuad_epochs=TRAIN_EPOCHS, batch_size=16,
)


In [ ]:
# Section 3d — Legal-BERT (CUAD)
# Re-tokenise with Legal-BERT's own vocabulary, then fine-tune
legal_bert_splits = prepare_chunked_splits(
    filtered_df, model_name="nlpaueb/legal-bert-base-uncased",
    max_length=512, stride=128, seed=42, sample_frac=SAMPLE_FRAC,
)
artifacts_legalbert = train_legal_bert_cuad(
    legal_bert_splits["train_dataset"], legal_bert_splits["val_dataset"],
    legal_bert_splits["train_examples"], legal_bert_splits["tokenizer"],
    legal_bert_splits["id_to_clause"],
    model_name="nlpaueb/legal-bert-base-uncased",
    epochs=TRAIN_EPOCHS, batch_size=16,
)


In [ ]:
# Section 3e — Longformer (CUAD)
# batch_size=2: reduced from 4; gradient_checkpointing (enabled in training.py)
# recomputes activations on the backward pass, trading ~15% extra compute for
# ~4x less activation memory — critical to fit 4096-token sequences on a 15 GB GPU.
artifacts_longformer = train_longformer_cuad(
    LF_SPLITS["train_dataset"], LF_SPLITS["val_dataset"],
    LF_SPLITS["train_examples"], LF_SPLITS["tokenizer"],
    LF_SPLITS["id_to_clause"],
    epochs=TRAIN_EPOCHS, batch_size=2,
)

# Section 3f — Legal-BERT → Longformer (CUAD)
artifacts_lf_lb = train_legalbert_longformer_cuad(
    LF_SPLITS["train_dataset"], LF_SPLITS["val_dataset"],
    LF_SPLITS["train_examples"], LF_SPLITS["tokenizer"],
    LF_SPLITS["id_to_clause"],
    epochs=TRAIN_EPOCHS, batch_size=2,
)

ALL_ARTIFACTS = [
    artifacts_tfidf, artifacts_bert_ledgar, artifacts_bert,
    artifacts_legalbert, artifacts_longformer, artifacts_lf_lb,
]


## Section 4 — Evaluation

In [ ]:
# Section 4 — Evaluate all models
from torch.utils.data import DataLoader as TDL

device = choose_device()
results_per_clause = {}
results_aggregate  = {}
tfidf_test_logits  = None
tfidf_test_labels  = None

for art in ALL_ARTIFACTS:
    # Get test logits
    if art.model_name == "TF-IDF + LR":
        test_titles = {r["contract_title"] for r in BERT_SPLITS["test_records"]}
        test_df_sub = filtered_df[filtered_df["contract_title"].isin(test_titles)]
        test_texts  = [grp["contract_text"].iloc[0]
                       for _, grp in test_df_sub.groupby("contract_title")]
        name_to_id  = {v: k for k, v in art.id_to_clause.items()}
        test_label_matrix = np.zeros((len(test_texts), len(art.id_to_clause)))
        for ti, (_, grp) in enumerate(test_df_sub.groupby("contract_title")):
            for _, row in grp.iterrows():
                if row["has_answer"] and row["clause_type"] in name_to_id:
                    test_label_matrix[ti, name_to_id[row["clause_type"]]] = 1.0
        eps = 1e-7
        probs = art.model.predict_proba(test_texts)
        p = np.clip(probs, eps, 1 - eps)
        test_logits = np.log(p / (1 - p))
        test_labels = test_label_matrix
        tfidf_test_logits = test_logits
        tfidf_test_labels = test_labels
    else:
        splits = LF_SPLITS if "Longformer" in art.model_name else BERT_SPLITS
        test_loader = TDL(splits["test_dataset"], batch_size=8, shuffle=False)
        test_logits, test_labels = collect_logits_and_labels(art.model, test_loader, device)

    # Tune per-clause thresholds on val logits, apply to test
    per_clause_thresholds = tune_per_clause_thresholds(
        art.val_logits, art.val_labels, art.id_to_clause
    )

    # Per-clause breakdown
    clause_df = compute_per_clause_metrics(
        test_logits, test_labels, per_clause_thresholds,
        art.id_to_clause, n_positive_train,
    )
    results_per_clause[art.model_name] = clause_df

    # Aggregate metrics
    agg = compute_aggregate_metrics(test_logits, test_labels, per_clause_thresholds, art.id_to_clause)
    results_aggregate[art.model_name] = agg
    print(f"{art.model_name:40s}  macro_F1={agg['macro_f1']:.4f}  micro_F1={agg['micro_f1']:.4f}")

# Aggregate comparison table
agg_df = pd.DataFrame(results_aggregate).T.reset_index().rename(columns={"index": "model"})
display(agg_df.sort_values("macro_f1", ascending=False))

# Per-clause breakdown for best model
best_model_name = agg_df.sort_values("macro_f1", ascending=False).iloc[0]["model"]
print(f"\nPer-clause breakdown — {best_model_name}:")
display(results_per_clause[best_model_name])

# Confusion matrices for all models
for art in ALL_ARTIFACTS:
    if art.model_name == "TF-IDF + LR":
        t_logits, t_labels = tfidf_test_logits, tfidf_test_labels
    else:
        splits = LF_SPLITS if "Longformer" in art.model_name else BERT_SPLITS
        t_loader = TDL(splits["test_dataset"], batch_size=8, shuffle=False)
        t_logits, t_labels = collect_logits_and_labels(art.model, t_loader, device)
    per_t = tune_per_clause_thresholds(art.val_logits, art.val_labels, art.id_to_clause)
    safe_name = art.model_name.replace(" ", "_").replace("→", "-")
    plot_confusion_matrix(
        t_logits, t_labels, per_t, art.id_to_clause,
        title=f"Confusion Matrix — {art.model_name}",
        save_path=f"figures/confusion_{safe_name}.png",
    )

# Model comparison chart
plot_model_comparison(results_per_clause, metric="f1", save_path="figures/model_comparison_f1.png")


## Section 5 — LLM Risk Summaries (Google Colab AI)

In [ ]:
# Section 5 — LLM risk summaries using Google Colab AI (no setup required)
# Uses Gemini 2.5 Flash — available free to all Colab users.
# Run  ai.list_models()  to see all available options.

from google.colab import ai

COLAB_MODEL = "google/gemini-2.5-flash"  # swap to "google/gemini-2.0-flash" for faster/lighter

RISK_PROMPT_TEMPLATE = """You are a contract risk analyst reviewing a flagged legal clause for a non-specialist business reader.

Clause type: {clause_type}
Contract text excerpt:
\"\"\"
{clause_text}
\"\"\"

Provide your analysis in exactly this format (no extra commentary):

RISK EXPLANATION: [2-3 sentences explaining what this clause means and why it matters, in plain English for a non-lawyer]
SEVERITY: [Low / Medium / High]
WATCH FOR: [One specific thing the reviewer should look out for or negotiate]"""


def generate_risk_summary(clause_type: str, clause_text: str) -> dict:
    response = ai.generate_text(
        model=COLAB_MODEL,
        prompt=RISK_PROMPT_TEMPLATE.format(
            clause_type=clause_type,
            clause_text=clause_text[:1500],
        ),
    )
    raw = response.text if hasattr(response, "text") else str(response)
    result = {"clause_type": clause_type, "clause_text": clause_text[:500], "raw_output": raw}
    for line in raw.split("\n"):
        if line.startswith("RISK EXPLANATION:"):
            result["risk_explanation"] = line.replace("RISK EXPLANATION:", "").strip()
        elif line.startswith("SEVERITY:"):
            result["severity"] = line.replace("SEVERITY:", "").strip()
        elif line.startswith("WATCH FOR:"):
            result["watch_for"] = line.replace("WATCH FOR:", "").strip()
    return result


# ── Build flagged clause list from best model ─────────────────────────────────
from torch.utils.data import DataLoader as TDL

best_art = next(a for a in ALL_ARTIFACTS if a.model_name == best_model_name)
splits_for_best = LF_SPLITS if "Longformer" in best_model_name else BERT_SPLITS

test_loader = TDL(splits_for_best["test_dataset"], batch_size=8, shuffle=False)
t_logits, _ = collect_logits_and_labels(best_art.model, test_loader, device)

probs = 1 / (1 + np.exp(-t_logits))
per_clause_t = tune_per_clause_thresholds(best_art.val_logits, best_art.val_labels, best_art.id_to_clause)

flagged_rows = []
for chunk_probs, example in zip(probs, splits_for_best["test_examples"]):
    for clause_id, clause_name in best_art.id_to_clause.items():
        t = per_clause_t.get(clause_name, 0.5)
        score = float(chunk_probs[clause_id])
        if score >= t:
            flagged_rows.append({
                "contract_title": example["contract_title"],
                "clause_type":    clause_name,
                "score":          score,
                "chunk_index":    example["chunk_index"],
                "chunk_text":     example["chunk_text"],
            })

flagged_df = pd.DataFrame(flagged_rows)
contract_agg = aggregate_contract_predictions(
    flagged_df[["contract_title", "clause_type", "score", "chunk_index"]]
)

flagged_df_dedup = (
    flagged_df.sort_values("score", ascending=False)
    .drop_duplicates(subset=["contract_title", "clause_type"])
)

sample_for_rating = flagged_df_dedup.sample(n=min(25, len(flagged_df_dedup)), random_state=42)

risk_summaries = []
for _, row in sample_for_rating.iterrows():
    summary = generate_risk_summary(row["clause_type"], row["chunk_text"])
    risk_summaries.append(summary)
    print(f"[{row['clause_type']}] Severity: {summary.get('severity', '?')}")

risk_df = pd.DataFrame(risk_summaries)
display(risk_df[["clause_type", "severity", "risk_explanation", "watch_for"]])

rating_template = risk_df[["clause_type", "chunk_text", "risk_explanation", "watch_for"]].copy()
rating_template["factual_accuracy_1_5"] = ""
rating_template["clarity_1_5"] = ""
rating_template.to_csv("risk_summary_ratings.csv", index=False)
print("Saved rating template to risk_summary_ratings.csv")
